# IA708 — Notebook 4 : Interprétabilité
## SHAP linéaire · Proxies · Permutation Importance

**Télécom Paris — Mastère IA Multimodale, 2026**

---

### Objectif de ce notebook

Après la performance (Notebook 2) et l'équité (Notebook 3), on répond ici à la question:
**"Pourquoi le modèle décide-t-il qu'un individu est risqué ?"**

Ce notebook fournit une lecture **traçable et auditable** des décisions :
1. **Interprétation locale et globale** avec SHAP (exact pour un modèle linéaire)
2. **Détection de proxies** d'attributs sensibles (genre/âge)
3. **Validation croisée** des importances via permutation importance

---

### Pourquoi cette étape est indispensable en IA responsable

- **Audit réglementaire / conformité** : un score de crédit doit être explicable
- **Détection de biais indirects** : retirer une variable sensible ne suffit pas si des proxies subsistent
- **Robustesse argumentative** : un modèle "performant" mais inexplicable est difficile à défendre

---

### Plan du notebook
0. Setup reproductible (mêmes données/split que Notebook 2)
1. SHAP linéaire : théorie + calcul exact
2. Visualisations SHAP : top features + beeswarm
3. Proxies : corrélations feature ↔ attribut sensible
4. Permutation importance : théorie + calcul
5. Comparaison SHAP vs permutation
6. Synthèse interprétabilité et recommandations



---
## 0. Setup (modèle baseline + reweighing)

On reconstruit ici un pipeline autonome, cohérent avec les notebooks précédents :
- même dataset (`german.data`) et même traduction des labels
- même split stratifié train/val/test
- même baseline logistique + modèle reweighing

### Garanties méthodologiques

- **Pas de leakage** : normalisation et encodage appris sur train uniquement
- **Comparabilité** : même prétraitement pour baseline et reweighing
- **Reproductibilité** : seeds fixées pour split/optimisation

Ce setup permet de comparer les méthodes d'interprétabilité dans des conditions contrôlées.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
plt.rcParams["figure.dpi"] = 120

# ── Données ───────────────────────────────────────────────────────────────────
COLUMNS = [
    "checking_status", "duration_in_month", "credit_history", "purpose",
    "credit_amount", "savings_account_bonds", "present_employment_since",
    "installment_rate", "personal_status_sex", "other_debtors_guarantors",
    "present_residence_since", "property", "age_in_years",
    "other_installment_plans", "housing", "number_of_existing_credits",
    "job", "number_of_people_liable", "telephone", "foreign_worker",
    "raw_target",
]
LABEL_MAP = {
    "checking_status": {"A11": "solde < 0 DM", "A12": "0 ≤ solde < 200 DM",
                        "A13": "solde ≥ 200 DM", "A14": "pas de compte courant"},
    "credit_history": {"A30": "aucun crédit ou tous remboursés", "A31": "tous crédits banque remboursés",
                       "A32": "crédits existants remboursés", "A33": "retards passés",
                       "A34": "compte critique"},
    "purpose": {"A40": "voiture (neuve)", "A41": "voiture (occasion)", "A42": "meubles",
                "A43": "radio/TV", "A44": "électroménager", "A45": "réparations",
                "A46": "éducation", "A47": "vacances", "A48": "reconversion",
                "A49": "business", "A410": "autres"},
    "savings_account_bonds": {"A61": "épargne < 100 DM", "A62": "100 ≤ épargne < 500 DM",
                              "A63": "500 ≤ épargne < 1000 DM", "A64": "épargne ≥ 1000 DM",
                              "A65": "inconnu / pas d'épargne"},
    "present_employment_since": {"A71": "sans emploi", "A72": "< 1 an",
                                  "A73": "1–4 ans", "A74": "4–7 ans", "A75": "≥ 7 ans"},
    "personal_status_sex": {"A91": "homme, divorcé/séparé", "A92": "femme, divorcée/mariée",
                            "A93": "homme, célibataire", "A94": "homme, marié/veuf",
                            "A95": "femme, célibataire"},
    "other_debtors_guarantors": {"A101": "aucun", "A102": "co-demandeur", "A103": "garant"},
    "property": {"A121": "immobilier", "A122": "épargne logement / assurance-vie",
                 "A123": "voiture ou autre", "A124": "inconnu / pas de propriété"},
    "other_installment_plans": {"A141": "banque", "A142": "magasins", "A143": "aucun"},
    "housing": {"A151": "locataire", "A152": "propriétaire", "A153": "hébergé gratuitement"},
    "job": {"A171": "sans emploi / non qualifié non-résident", "A172": "non qualifié résident",
            "A173": "employé qualifié", "A174": "management / hautement qualifié"},
    "telephone": {"A191": "aucun", "A192": "oui (au nom du client)"},
    "foreign_worker": {"A201": "oui", "A202": "non"},
}
raw = pd.read_csv("data/raw/german.data", sep=r"\s+", header=None, names=COLUMNS)
for col, mapping in LABEL_MAP.items():
    raw[col] = raw[col].astype(str).map(mapping).fillna(raw[col].astype(str))
raw["default"] = (raw["raw_target"] == 2).astype(int)
GENDER_MAP = {
    "homme, divorcé/séparé": "male", "homme, célibataire": "male",
    "homme, marié/veuf": "male",
    "femme, divorcée/mariée": "female", "femme, célibataire": "female",
}
raw["gender"]    = raw["personal_status_sex"].map(GENDER_MAP).fillna("unknown")
raw["age_group"] = np.where(raw["age_in_years"] > 25, "older", "younger")
sensitive  = {"gender": raw["gender"].values, "age": raw["age_group"].values}
PRIVILEGED = {"gender": "male", "age": "older"}
features = raw.drop(columns=[
    "raw_target", "default", "personal_status_sex", "age_in_years",
    "gender", "age_group"
])
NUMERIC = ["duration_in_month", "credit_amount", "installment_rate",
           "present_residence_since", "number_of_existing_credits",
           "number_of_people_liable"]
CATEG = [c for c in features.columns if c not in NUMERIC]
y = raw["default"].values

rng = np.random.default_rng(42)
def stratified_split(y, ratios=(0.6, 0.2, 0.2)):
    idx_train, idx_val, idx_test = [], [], []
    for label in np.unique(y):
        ix = np.flatnonzero(y == label); rng.shuffle(ix)
        n = len(ix); n_tr = int(round(n * ratios[0])); n_va = int(round(n * ratios[1]))
        idx_train.extend(ix[:n_tr]); idx_val.extend(ix[n_tr:n_tr+n_va]); idx_test.extend(ix[n_tr+n_va:])
    return np.array(idx_train), np.array(idx_val), np.array(idx_test)

tr, va, te = stratified_split(y)

class Preprocessor:
    def fit(self, df):
        self.num_means = df[NUMERIC].astype(float).mean()
        self.num_stds  = df[NUMERIC].astype(float).std(ddof=1).replace(0, 1)
        self.cat_levels, self.dummy_cols = {}, {}
        for c in CATEG:
            vals = sorted(df[c].astype(str).unique())
            self.cat_levels[c] = vals; self.dummy_cols[c] = [f"{c}_{v}" for v in vals]
        self.feature_names = list(NUMERIC)
        for c in CATEG: self.feature_names.extend(self.dummy_cols[c])
        return self
    def transform(self, df):
        parts = [(df[NUMERIC].astype(float) - self.num_means) / self.num_stds]
        parts[0] = parts[0].reset_index(drop=True)
        for c in CATEG:
            cat = pd.Categorical(df[c].astype(str), categories=self.cat_levels[c])
            dum = pd.get_dummies(cat, prefix=c, dtype=float).reindex(columns=self.dummy_cols[c], fill_value=0.0)
            parts.append(dum.reset_index(drop=True))
        out = pd.concat(parts, axis=1); out.columns = self.feature_names
        return out.values

prep = Preprocessor().fit(features.iloc[tr])
X_tr = prep.transform(features.iloc[tr])
X_va = prep.transform(features.iloc[va])
X_te = prep.transform(features.iloc[te])
y_tr, y_va, y_te = y[tr], y[va], y[te]

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -50, 50)))

def train_logreg(X, y, X_val, y_val, lr=0.03, l2=0.01, epochs=3500,
                 patience=300, sample_weight=None):
    n, d = X.shape
    w = np.zeros(d); p0 = np.clip(y.mean(), 1e-4, 1-1e-4)
    b = float(np.log(p0/(1-p0)))
    sw = np.ones(n) if sample_weight is None else sample_weight*(n/sample_weight.sum())
    mw, vw, mb, vb = np.zeros(d), np.zeros(d), 0.0, 0.0
    best_w, best_b, best_loss = w.copy(), b, np.inf; stale = 0; history = []
    for ep in range(1, epochs+1):
        p_hat = sigmoid(X@w+b); err = (p_hat-y)*sw
        gw = X.T@err/n+l2*w; gb = err.mean()
        mw=0.9*mw+0.1*gw; vw=0.999*vw+0.001*gw**2
        mb=0.9*mb+0.1*gb; vb=0.999*vb+0.001*gb**2
        mwh=mw/(1-0.9**ep); vwh=vw/(1-0.999**ep)
        mbh=mb/(1-0.9**ep); vbh=vb/(1-0.999**ep)
        w -= lr*mwh/(np.sqrt(vwh)+1e-8); b -= lr*mbh/(np.sqrt(vbh)+1e-8)
        p_val = sigmoid(X_val@w+b)
        vl = -np.mean(y_val*np.log(np.clip(p_val,1e-8,1-1e-8))+(1-y_val)*np.log(np.clip(1-p_val,1e-8,1-1e-8)))
        history.append(vl)
        if vl+1e-6 < best_loss: best_loss=vl; best_w,best_b=w.copy(),b; stale=0
        else:
            stale += 1
            if stale >= patience: break
    return best_w, best_b, history

def predict_scores(X, w, b): return sigmoid(X@w+b)
def best_threshold(y_true, scores, n_c=81):
    best_t, best_ba = 0.5, 0.0
    for t in np.linspace(0.1, 0.9, n_c):
        pred = (scores>=t).astype(int)
        tp=((pred==1)&(y_true==1)).sum(); tn=((pred==0)&(y_true==0)).sum()
        ba=0.5*(tp/max((y_true==1).sum(),1)+tn/max((y_true==0).sum(),1))
        if ba > best_ba: best_ba, best_t = ba, t
    return best_t

def auc_roc(y, scores):
    pos, neg = (y==1).sum(), (y==0).sum()
    if pos==0 or neg==0: return float("nan")
    ranks = pd.Series(scores).rank(method="average").values
    return (ranks[y==1].sum()-pos*(pos+1)/2)/(pos*neg)

def reweighing_weights(y, sens):
    n = len(y); weights = np.ones(n)
    p_y=pd.Series(y).value_counts(normalize=True); p_s=pd.Series(sens).value_counts(normalize=True)
    joint=pd.DataFrame({"y":y,"s":sens}).groupby(["s","y"]).size()/n
    for i in range(n):
        key=(sens[i],y[i]); pj=joint.get(key,0)
        if pj>0: weights[i]=p_s[sens[i]]*p_y[y[i]]/pj
    return weights/weights.mean()

# Entraînement baseline et reweighing (genre)
w_base, b_base, _ = train_logreg(X_tr, y_tr, X_va, y_va)
scores_va  = predict_scores(X_va, w_base, b_base)
scores_te  = predict_scores(X_te, w_base, b_base)
thr_base   = best_threshold(y_va, scores_va)
preds_base = (scores_te >= thr_base).astype(int)

s_tr_gender = sensitive["gender"][tr]
w_rw = reweighing_weights(y_tr, s_tr_gender)
w_fair, b_fair, _ = train_logreg(X_tr, y_tr, X_va, y_va, sample_weight=w_rw)

print(f"Setup OK — Baseline AUC = {auc_roc(y_te, scores_te):.4f}")

---
## 1. SHAP linéaire : théorie et interprétation

### 1.1 Rappel Shapley

Les valeurs SHAP (Lundberg & Lee, 2017) attribuent à chaque feature une contribution additive:

$$f(x)=\phi_0+\sum_{i=1}^{d}\phi_i(x)$$

avec $\phi_0=\mathbb{E}[f(X)]$.

Axiomes utiles en pratique:
- **Efficacité** : la somme des contributions explique exactement la prédiction
- **Symétrie** : deux features équivalentes reçoivent la même contribution
- **Nul** : une feature sans effet a contribution nulle

### 1.2 Cas linéaire (exact)

Pour $f(x)=w^\top x+b$, on a une forme fermée:

$$\phi_i(x)=w_i\cdot(x_i-\mathbb{E}_{train}[x_i])$$

Ici, SHAP est **exact** (pas d'approximation Monte Carlo).

### 1.3 Attention à l'échelle

Le modèle travaille en **log-odds**. Donc:
- SHAP > 0 : pousse vers plus de risque (défaut)
- SHAP < 0 : pousse vers moins de risque

Pour lire en probabilité, il faut repasser par la sigmoïde; les effets ne sont alors plus strictement additifs.

### 1.4 Agrégation des catégorielles encodées

Une variable catégorielle encodée en one-hot produit plusieurs colonnes.
On regroupe via:

$$\phi_{feature}(x)=\sum_{k\in dummies(feature)}\phi_k(x)$$

Cela rend l'analyse lisible au niveau métier (feature d'origine).



In [ ]:
def compute_shap_by_feature(w, X_test, X_train_mean):
    """
    Calcule les valeurs SHAP exactes pour un modèle linéaire.

    Paramètres
    ----------
    w             : vecteur de poids du modèle (d,)
    X_test        : données test encodées (n, d)
    X_train_mean  : moyenne des features sur le train (d,) — référence SHAP

    Retourne
    --------
    shap_vals  : valeurs SHAP par feature encodée (n, d)
    shap_by_feature : DataFrame triée par mean|SHAP| dans l'espace original
    """
    # Formule exacte : phi_i = w_i * (x_i - E[x_i])
    shap_vals = (X_test - X_train_mean) * w  # (n, d)

    # Agrégation par feature originale
    feature_importances = {}

    for col in NUMERIC:
        idx = prep.feature_names.index(col)
        feature_importances[col] = np.abs(shap_vals[:, idx]).mean()

    for col in CATEG:
        indices = [prep.feature_names.index(c) for c in prep.dummy_cols[col]
                   if c in prep.feature_names]
        if indices:
            feature_importances[col] = np.abs(shap_vals[:, indices].sum(axis=1)).mean()

    shap_by_feature = pd.DataFrame({
        "feature": list(feature_importances.keys()),
        "mean_abs_shap": list(feature_importances.values())
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    return shap_vals, shap_by_feature

# Calcul pour baseline et reweighing
X_train_mean = X_tr.mean(axis=0)  # référence = moyenne du train

shap_vals_base, shap_df_base = compute_shap_by_feature(w_base, X_te, X_train_mean)
shap_vals_fair, shap_df_fair = compute_shap_by_feature(w_fair, X_te, X_train_mean)

print("Top 10 features (SHAP baseline) :")
print(shap_df_base.head(10).to_string(index=False))

---
## 2. Visualisations SHAP : guide de lecture

### 2.1 Bar chart (importance globale)

Le bar chart classe les features par **mean |SHAP|**:
- plus la barre est grande, plus la feature influence les décisions globalement
- comparaison baseline vs reweighing: on voit quelles variables changent de poids

### 2.2 Beeswarm (distribution locale)

Le beeswarm montre, pour chaque feature:
- la distribution des contributions individuelles (dispersion)
- le signe des contributions (gauche = baisse du risque, droite = hausse du risque)
- la couleur (faible/forte valeur de feature), utile pour comprendre la direction de l'effet

### 2.3 Points d'attention

- Une feature peut avoir une moyenne modérée mais des effets extrêmes sur certains individus
- Les patterns très asymétriques signalent souvent des interactions implicites ou sous-groupes
- L'interprétation locale doit rester cohérente avec l'expertise métier (credit risk)



In [ ]:
# ── Bar chart : Top features par mean|SHAP| ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
n_show = 12

for ax, (shap_df, label, color) in zip(axes, [
    (shap_df_base, "Baseline", "#d97c3a"),
    (shap_df_fair, "Reweighing", "#4CAF50")
]):
    top = shap_df.head(n_show).sort_values("mean_abs_shap")
    ax.barh(top["feature"], top["mean_abs_shap"], color=color, alpha=0.85)
    ax.set_xlabel("Mean |SHAP value| (log-odds)")
    ax.set_title(f"SHAP — {label}")
    for i, (_, row) in enumerate(top.iterrows()):
        ax.text(row["mean_abs_shap"] + 0.002, i,
                f"{row['mean_abs_shap']:.3f}", va="center", fontsize=8)

plt.suptitle("Top features selon SHAP — baseline vs reweighing",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Beeswarm SHAP : distribution des valeurs par feature ─────────────────────
# Pour chaque feature (dans l'espace encodé), on trace les valeurs SHAP
# colorées par la valeur de la feature (rouge = haute, bleu = basse)

top_features = shap_df_base.head(8)["feature"].tolist()

fig, axes = plt.subplots(2, 4, figsize=(17, 8))
axes = axes.flatten()

for ax, feat in zip(axes, top_features):
    # Récupérer les valeurs SHAP pour cette feature
    if feat in NUMERIC:
        idx = prep.feature_names.index(feat)
        shap_f = shap_vals_base[:, idx]
        feat_vals = X_te[:, idx]  # valeur standardisée
    else:
        indices = [prep.feature_names.index(c) for c in prep.dummy_cols[feat]
                   if c in prep.feature_names]
        shap_f = shap_vals_base[:, indices].sum(axis=1)
        feat_vals = shap_f  # proxy : utiliser le SHAP comme couleur

    # Normaliser les couleurs
    norm_vals = (feat_vals - feat_vals.min()) / (feat_vals.max() - feat_vals.min() + 1e-8)
    colors_scatter = cm.RdBu_r(norm_vals)

    # Beeswarm (jitter vertical)
    jitter = np.random.default_rng(0).uniform(-0.3, 0.3, len(shap_f))
    ax.scatter(shap_f, jitter, c=colors_scatter, s=10, alpha=0.6)
    ax.axvline(0, color="black", linewidth=0.8, alpha=0.5)
    ax.set_yticks([])
    ax.set_xlabel("SHAP value")
    ax.set_title(feat.replace("_", " "), fontsize=9, fontweight="bold")

plt.suptitle("Beeswarm SHAP (rouge = valeur haute, bleu = valeur basse)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Lecture : SHAP > 0 → augmente le risque de défaut")
print("         SHAP < 0 → réduit le risque de défaut")

---
## 3. Détection de proxies via SHAP / features

### 3.1 Définition opérationnelle

Un **proxy** est une variable non sensible corrélée à un attribut sensible,
et qui peut donc transporter un biais indirect.

Exemples classiques sur German Credit:
- `duration_in_month` comme proxy potentiel de l'âge
- certaines modalités de `purpose` comme proxy potentiel du genre

### 3.2 Méthode utilisée ici

On calcule une corrélation absolue entre feature (ou agrégation one-hot) et attribut sensible binaire.
On utilise des seuils heuristiques pour prioriser l'audit:
- |corr| > 0.15 : proxy fort (à investiguer)
- 0.08 < |corr| <= 0.15 : signal faible à moyen

### 3.3 Limites de la méthode

- Corrélation != causalité
- Une corrélation faible univariée n'exclut pas un proxy multivarié
- Les petits effectifs de sous-groupes augmentent la variance des estimations

Conclusion pratique: cette étape sert à **prioriser** les variables à approfondir, pas à prouver seule une discrimination.



In [ ]:
# Corrélation entre SHAP de chaque feature et l'attribut sensible
X_te_df = pd.DataFrame(X_te, columns=prep.feature_names)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, attr in zip(axes, ["gender", "age"]):
    s_num = pd.Series(sensitive[attr][te]).map(
        {"male": 1, "female": 0, "older": 1, "younger": 0}
    ).astype(float).values

    # Corrélation par feature originale
    proxy_scores = {}
    for feat in list(NUMERIC) + list(CATEG):
        if feat in NUMERIC:
            idx = prep.feature_names.index(feat)
            vals = X_te[:, idx]
        else:
            indices = [prep.feature_names.index(c) for c in prep.dummy_cols[feat]
                       if c in prep.feature_names]
            vals = X_te[:, indices].sum(axis=1)
        corr = np.corrcoef(vals, s_num)[0, 1]
        proxy_scores[feat] = abs(corr)

    proxy_df = pd.Series(proxy_scores).sort_values(ascending=False).head(10)
    colors_proxy = ["#f44336" if v > 0.15 else "#FF9800" if v > 0.08 else "#4CAF50"
                    for v in proxy_df.values]
    ax.barh(proxy_df.index[::-1], proxy_df.values[::-1], color=colors_proxy[::-1], alpha=0.85)
    ax.axvline(0.15, color="red", linestyle="--", alpha=0.7, label="Seuil proxy (0.15)")
    ax.axvline(0.08, color="orange", linestyle=":", alpha=0.7, label="Seuil faible (0.08)")
    ax.set_xlabel("|Corrélation| avec l'attribut sensible")
    ax.set_title(f"Proxies potentiels — {attr}")
    ax.legend(fontsize=8)
    for i, v in enumerate(proxy_df.values[::-1]):
        ax.text(v + 0.002, i, f"{v:.3f}{'  PROXY' if v > 0.15 else ''}",
                va="center", fontsize=8,
                color="red" if v > 0.15 else "black")

plt.suptitle("Détection de proxies (corrélation feature–attribut sensible)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Importance par permutation

### 4.1 Idée

On casse la relation feature-cible en permutant une feature, puis on mesure la chute d'AUC:

$$Imp(j)=AUC_{orig}-\frac{1}{R}\sum_{r=1}^{R}AUC(\tilde{X}_{j,r})$$

- grande chute => feature importante
- chute faible => feature redondante/peu utile

### 4.2 Pourquoi compléter SHAP avec permutation

SHAP (linéaire) est très interprétable, mais permutation apporte une validation
au niveau performance prédictive.

- SHAP: "contribution du modèle"
- Permutation: "combien la performance souffre si on détruit cette info"

### 4.3 Limites

- En présence de colinéarité, permutation peut sous-estimer certaines features
- Coût computationnel plus élevé (répétitions)
- Dépend de la métrique choisie (ici AUC)

On répète `R=10` permutations pour réduire le bruit de Monte Carlo.



In [ ]:
def permutation_importance(w, b, X_test, y_test, R=10, seed=42):
    """
    Calcule l'importance par permutation pour chaque feature originale.

    Paramètres
    ----------
    R : nombre de répétitions par feature

    Retourne
    --------
    DataFrame avec les features triées par chute d'AUC moyenne
    """
    rng_p = np.random.default_rng(seed)
    base_auc = auc_roc(y_test, predict_scores(X_test, w, b))

    all_features = list(NUMERIC) + list(CATEG)
    importances = {}

    for feat in all_features:
        if feat in NUMERIC:
            feat_indices = [prep.feature_names.index(feat)]
        else:
            feat_indices = [prep.feature_names.index(c)
                            for c in prep.dummy_cols[feat]
                            if c in prep.feature_names]

        drops = []
        for _ in range(R):
            X_shuf = X_test.copy()
            # Mélanger toutes les colonnes correspondant à cette feature
            perm = rng_p.permutation(len(X_test))
            for idx in feat_indices:
                X_shuf[:, idx] = X_test[perm, idx]
            drops.append(base_auc - auc_roc(y_test, predict_scores(X_shuf, w, b)))

        importances[feat] = np.mean(drops)

    return pd.DataFrame({
        "feature": list(importances.keys()),
        "auc_drop": list(importances.values())
    }).sort_values("auc_drop", ascending=False).reset_index(drop=True)

print("Calcul de l'importance par permutation...")
perm_df_base = permutation_importance(w_base, b_base, X_te, y_te)
perm_df_fair = permutation_importance(w_fair, b_fair, X_te, y_te)
print("Terminé.")
print("\nTop 10 (baseline) :")
print(perm_df_base.head(10).to_string(index=False))

---
## 5. Comparaison SHAP vs Permutation

Cette section compare les rangs des features selon deux logiques:
- **SHAP** : importance structurelle dans le modèle
- **Permutation** : importance mesurée par perte de performance

### Comment lire les divergences

- Convergence des top features: signal robuste
- Divergence forte sur une feature: possible colinéarité, effet de groupe, ou sensibilité au prétraitement

Le tableau de concordance des rangs (|Δ rang|) aide à identifier les features stables
versus celles à investiguer davantage.



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
n_top = 10

# ── Côte à côte SHAP vs Permutation ──────────────────────────────────────────
for ax, (shap_df, perm_df, label) in zip(axes, [
    (shap_df_base, perm_df_base, "Baseline"),
    (shap_df_fair, perm_df_fair, "Reweighing")
]):
    top_shap = shap_df.head(n_top)["feature"].tolist()
    top_perm = perm_df.head(n_top)["feature"].tolist()
    all_feats = list(dict.fromkeys(top_shap + top_perm))[:n_top + 3]

    shap_vals_map = dict(zip(shap_df["feature"], shap_df["mean_abs_shap"]))
    perm_vals_map = dict(zip(perm_df["feature"], perm_df["auc_drop"]))

    # Normaliser pour afficher côte à côte
    shap_norm = [shap_vals_map.get(f, 0) / max(shap_df["mean_abs_shap"]) for f in all_feats]
    perm_norm = [perm_vals_map.get(f, 0) / max(perm_df["auc_drop"].clip(lower=0)) for f in all_feats]

    x = np.arange(len(all_feats))
    w_bar = 0.35
    ax.bar(x - w_bar/2, shap_norm, w_bar, color="#d97c3a", alpha=0.85, label="SHAP (normalisé)")
    ax.bar(x + w_bar/2, perm_norm, w_bar, color="#2f6f9f", alpha=0.85, label="Perm. Imp. (normalisé)")
    ax.set_xticks(x)
    ax.set_xticklabels([f[:20] for f in all_feats], rotation=35, ha="right", fontsize=8)
    ax.set_ylabel("Importance normalisée")
    ax.set_title(f"{label} — SHAP vs Permutation")
    ax.legend(fontsize=9)

plt.suptitle("Comparaison des méthodes d'interprétabilité",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Rang de concordance
print("\n=== Concordance des rangs (baseline) ===")
shap_ranks = {f: i+1 for i, f in enumerate(shap_df_base["feature"])}
perm_ranks = {f: i+1 for i, f in enumerate(perm_df_base["feature"])}
print(f"{'Feature':<35} {'Rang SHAP':>10} {'Rang Perm':>10} {'|Δ rang|':>10}")
for feat in shap_df_base.head(8)["feature"]:
    sr, pr = shap_ranks.get(feat, 99), perm_ranks.get(feat, 99)
    print(f"  {feat:<33} {sr:>10} {pr:>10} {abs(sr-pr):>10}")

---
## 6. Synthèse interprétabilité

### Ce qu'il faut documenter après exécution

1. **Top drivers** (SHAP + permutation) et justification métier
2. **Proxies potentiels** détectés (niveau de risque faible/moyen/fort)
3. **Différences baseline vs reweighing** sur les importances
4. **Convergences/divergences SHAP vs permutation**

### Checklist de décision

- Les 3 premières features sont-elles plausibles pour un analyste crédit ?
- Existe-t-il des proxies sensibles avec |corr| > 0.15 ?
- Le reweighing réduit-il l'importance de proxies problématiques ?
- Les conclusions sont-elles stables entre méthodes d'explicabilité ?

### Message clé

Un modèle interprétable n'est pas seulement "expliqué":
il est **auditable**, **cohérent métier**, et **actionnable** pour corriger les risques de biais.

La robustesse sous perturbation est traitée en détail dans Notebook 5.

